In [1]:
import firecloud.api as fapi
# import subprocess
import pandas as pd
# import importlib
# import io
# # from io import StringIO
# import dumper

In [2]:
import importlib
import os
import subprocess
import igvf_and_terra_api_tools as api_tools
import portal_to_terra_input_from_anaset as p2t_tools
import terra_to_portal_posting as t2p_tools

In [3]:
import igvf_utils as iu
from igvf_utils.connection import Connection

In [218]:
importlib.reload(t2p_tools)

<module 'terra_to_portal_posting' from '/Users/zheweishen/IGVF/IGVF_Repos/sc-pipeline-management/src/terra_to_portal_posting.py'>

### Params

In [31]:
# Set up IGVF query API (Sandbox or Production)
igvf_api_sandbox = api_tools.get_igvf_auth_and_api(igvf_site='sandbox')
igvf_api_production = api_tools.get_igvf_auth_and_api(igvf_site='production')

# IGVF util
iu_conn = Connection(igvf_mode='sandbox', submission=True)

# If FireCloud session somehow expires
fapi._set_session()

2024-12-12 14:55:56,997:iu_debug:	submission=True: In submission mode.


DEBUG:iu_debug:submission=True: In submission mode.


In [5]:
# Set params for Terra
terra_namespace = 'DACC_ANVIL'
terra_workspace = 'Playground for IGVF Single-Cell Data Processing'

# Set params for IGVF
prod_analysis_sets = ['IGVFDS5839UBFZ', 'IGVFDS9520PLLU']
sandbox_analysis_sets = ['TSTDS33660419', 'TSTDS74356276']

### Run Portal input and to Terra

#### Test out thoughts

In [4]:
t2p_tools.get_file_extension('gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7ad1a5/submissions/f72bac2f-e428-467c-ac78-4a4919e04103/multiome_pipeline/bae29912-fb64-4614-b198-5f99d27248ef/call-rna/wf_rna/96adc130-7173-4b3f-8162-113618a2c60d/call-kb/Dataset510xmultiomLane3.rna.align.kb.hg38.count_matrix.h5ad')

'rna'

In [73]:
foo = p2t_tools.SingleCellInputBuilder(analysis_set_acc=['IGVFDS9520PLLU'], igvf_api=igvf_api_production)

In [74]:
foo.get_seqfile_uris_and_seqspec_by_assays_and_runs()
# foo.get_data_access_auth()

In [75]:
foo.data

{'analysis_set_acc': '',
 'lab': '',
 'awards': '',
 'controlled_access': 'false',
 'atac_MeaSetIDs': [],
 'rna_MeaSetIDs': ['IGVFDS5205JRNX'],
 'taxa': [],
 'atac_read1_accessions': [],
 'atac_read2_accessions': [],
 'rna_read1_accessions': ['IGVFFI0541TEFJ', 'IGVFFI7011HTGX'],
 'rna_read2_accessions': ['IGVFFI0459FHFI', 'IGVFFI0090YFWB'],
 'seqspec_accessions': {'IGVFFI0535LWJW', 'IGVFFI3228LOOM'},
 'atac_read1': [],
 'atac_read2': [],
 'rna_read1': ['https://api.data.igvf.org/sequence-files/IGVFFI0541TEFJ/@@download/IGVFFI0541TEFJ.fastq.gz',
  'https://api.data.igvf.org/sequence-files/IGVFFI7011HTGX/@@download/IGVFFI7011HTGX.fastq.gz'],
 'rna_read2': ['https://api.data.igvf.org/sequence-files/IGVFFI0459FHFI/@@download/IGVFFI0459FHFI.fastq.gz',
  'https://api.data.igvf.org/sequence-files/IGVFFI0090YFWB/@@download/IGVFFI0090YFWB.fastq.gz'],
 'seqspec_urls': {'https://api.data.igvf.org/configuration-files/IGVFFI0535LWJW/@@download/IGVFFI0535LWJW.yaml.gz',
  'https://api.data.igvf.org/c

In [78]:
bar = foo.data['atac_read1_accessions'] + foo.data['atac_read2_accessions'] + foo.data['rna_read1_accessions'] + foo.data['rna_read2_accessions']
bar

['IGVFFI0541TEFJ', 'IGVFFI7011HTGX', 'IGVFFI0459FHFI', 'IGVFFI0090YFWB']

In [80]:
for seqfile_acc in bar:
    print(seqfile_acc)
    seqfile_item = igvf_api_production.sequence_files(accessions=[seqfile_acc])

IGVFFI0541TEFJ


ValidationError: 1 validation error for sequence_files
accessions
  Unexpected keyword argument [type=unexpected_keyword_argument, input_value=['IGVFFI0541TEFJ'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.9/v/unexpected_keyword_argument

In [5]:
igvf_api_production.sequence_files(accession=['IGVFFI0541TEFJ'])

SequenceFileResults(graph=[SequenceFile(externally_hosted=False, external_host_url=None, controlled_access=False, anvil_url=None, release_timestamp='2024-05-08T23:38:37.124241+00:00', documents=None, lab='/labs/ali-mortazavi/', award='/awards/HG012077/', accession='IGVFFI0541TEFJ', alternate_accessions=None, collections=None, status='released', revoke_detail=None, schema_version='14', uuid='efed992e-b4c4-4a76-8152-3ddbaf4203db', notes=None, aliases=None, creation_timestamp='2023-08-25T19:52:07.000874+00:00', submitted_by='/users/ebd1025b-e3ab-4e51-bfcd-1415f278281a/', submitter_comment=None, description=None, analysis_step_version=None, content_md5sum='7c8ea84f5d4ac98524055f519a487886', content_type='reads', dbxrefs=None, derived_from=None, derived_manually=None, file_format='fastq', file_format_specifications=None, file_set='/measurement-sets/IGVFDS5205JRNX/', file_size=12202759123, md5sum='aa7dc0237ddb4349805abfb2f8c73012', submitted_file_name='igvf_b01/next2/igvf_b01_B01_D_next_N2_R

In [ ]:
igvf_api_production.sequence_files(accession=)

#### Full test run

In [28]:
# Get input file
igvf_input_table = p2t_tools.main(query_analysis_set_accs=sandbox_analysis_sets, 
                                  igvf_api=igvf_api_sandbox
                                 )
igvf_input_table

,analysis_set_acc,atac_MeaSetIDs,rna_MeaSetIDs,taxa,atac_read1_accessions,atac_read2_accessions,atac_barcode_accessions,rna_read1_accessions,rna_read2_accessions,seqspec_accessions,atac_read1,atac_read2,rna_read1,rna_read2,seqspec_urls
0,TSTDS33660419,[TSTDS87815319],[TSTDS22634158],[Homo sapiens],"[TSTFI68061834, TSTFI78417402]","[TSTFI18171101, TSTFI58997785]",[],"[TSTFI68220577, TSTFI26582203]","[TSTFI38034800, TSTFI83066582]",[],[https://api.data.igvf.org/sequence-files/TSTF...,[https://api.data.igvf.org/sequence-files/TSTF...,[https://api.data.igvf.org/sequence-files/TSTF...,[https://api.data.igvf.org/sequence-files/TSTF...,[]
1,TSTDS74356276,[TSTDS87815319],[TSTDS22634158],[Homo sapiens],"[TSTFI68061834, TSTFI78417402]","[TSTFI18171101, TSTFI58997785]",[],"[TSTFI68220577, TSTFI26582203]","[TSTFI38034800, TSTFI83066582]",[],[https://api.data.igvf.org/sequence-files/TSTF...,[https://api.data.igvf.org/sequence-files/TSTF...,[https://api.data.igvf.org/sequence-files/TSTF...,[https://api.data.igvf.org/sequence-files/TSTF...,[]


In [32]:
# Upload to Terra
api_tools.upload_portal_input_tsv_to_terra(terra_namespace=terra_namespace, 
                                           terra_workspace=terra_workspace, 
                                           terra_etype='DACC_tester_from_sandbox_2', # What you want to call it
                                           porta_input_table=igvf_input_table, 
                                           verbose=False
                                          )

### Push from Terra to Portal

In [25]:
# For posting (this mock table mixes the input table because it has IGVF accessions and some output from Team 2 because I need files to try posting)
terra_etype = 'DACC_mockTeam_2_tester'
Sandbox_test_analysis_set = 'TSTDS33660419'

In [26]:
# Generate the input data table for Terra
terra_push_example = api_tools.get_terra_tsv_data_table(terra_namespace=terra_namespace,
                                                          terra_workspace=terra_workspace,
                                                          terra_etype=terra_etype
                                                         )

In [27]:
terra_push_example.head(2)

,entity:DACC_mockTeam_2_tester_id,analysis_set_acc,atac_bam,atac_bam_log,atac_barcode_accessions,ATAC_barcode_offset,atac_chromap_barcode_metadata,atac_filter_fragments,atac_filter_fragments_index,atac_read1_accessions,...,rna_mtx_tar,rna_mtxs_h5ad,rna_read1_accessions,rna_read2_accessions,seqspec,seqspec_atac_index,seqspec_atac_onlist,seqspec_rna_index,seqspec_rna_onlist,Subpool
0,TSTDS33660419,TSTDS33660419,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,/Users/zheweishen/IGVF/single_cell_fg/Portal_i...,"[""syn52227976"",""syn52227995"",""syn52228015"",""sy...",8,/Users/zheweishen/IGVF/single_cell_fg/Portal_i...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,"['TSTFI68061834', 'TSTFI78417402']",...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,"['TSTFI68220577', 'TSTFI26582203']","['TSTFI38034800', 'TSTFI83066582']",https://raw.githubusercontent.com/detrout/y2av...,"bc:8:23:-,r1:0:49,r2:0:49",gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,"0,0,16:0,16,28:1,0,90",gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,Dataset510xmultiomLane1
1,TSTDS74356276,TSTDS74356276,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,/Users/zheweishen/IGVF/single_cell_fg/Portal_i...,"[""syn52228073"",""syn52228091"",""syn52228139"",""sy...",8,/Users/zheweishen/IGVF/single_cell_fg/Portal_i...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,"['TSTFI68061834', 'TSTFI78417402']",...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,"['TSTFI68220577', 'TSTFI26582203']","['TSTFI38034800', 'TSTFI83066582']",https://raw.githubusercontent.com/detrout/y2av...,"bc:8:23:-,r1:0:49,r2:0:49",gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,"0,0,16:0,16,28:1,0,90",gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7...,Dataset510xmultiomLane2


#### Try posting (RNA and ATAC)

In [174]:
# Get lab and award
tester_award, tester_lab = t2p_tools.get_lab_and_award(terra_data_record=sub_terra_push_series, igvf_api=igvf_api_sandbox)

# Get sequence file controlled_access info
tester_seqfiles = t2p_tools.get_seqfile_accs_from_table(terra_data_record=sub_terra_push_series, seqfile_acc_cols=['rna_read1_accessions', 'rna_read2_accessions'])

In [204]:
# Try post all RNA output
rna_post_log = t2p_tools.post_all_rna_data_to_portal(terra_data_record=sub_terra_push_series, award=tester_award, lab=tester_lab, igvf_utils_api=iu_conn, upload_file=False)

2024-12-03 15:57:44,797:iu_debug:	
IN post().


DEBUG:iu_debug:
IN post().


2024-12-03 15:57:45,682:iu_debug:	Validating the payload against the schema


DEBUG:iu_debug:Validating the payload against the schema


2024-12-03 15:57:45,733:iu_debug:	<<<<<< POST matrix_file record ansuman-satpathy:TSTDS33660419_rna_aggregated_counts_h5ad_uniform-pipeline To IGVF database with URL https://api.sandbox.igvf.org/matrix_file and this payload:

{
  "aliases": [
    "ansuman-satpathy:TSTDS33660419_rna_aggregated_counts_h5ad_uniform-pipeline"
  ],
  "award": "/awards/HG012076/",
  "content_type": "sparse gene count matrix",
  "derived_from": [
    "TSTFI68220577",
    "TSTFI26582203",
    "TSTFI38034800",
    "TSTFI83066582"
  ],
  "file_format": "h5ad",
  "file_set": "TSTDS33660419",
  "file_size": 1273491608,
  "lab": "/labs/ansuman-satpathy/",
  "md5sum": "8cfc749f2a47c4020e50c4acd9ba786b",
  "principal_dimension": "cell",
  "reference_files": [
    "TSTFI36924773"
  ],
  "secondary_dimensions": [
    "gene"
  ],
  "submitted_file_name": "gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7ad1a5/submissions/f72bac2f-e428-467c-ac78-4a4919e04103/multiome_pipeline/8aa086d8-3230-4e7d-aa6b-a2e70bfa2aad/call-rna/wf

...

DEBUG:iu_debug:Success.


2024-12-03 15:57:55,474:iu_debug:	Object posted with identifier: TSTFI23372307


DEBUG:iu_debug:Object posted with identifier: TSTFI23372307
INFO:iu_post:ansuman-satpathy:TSTDS33660419_rna_kb_output_uniform-pipeline		TSTFI23372307


2024-12-03 15:57:55,477:iu_debug:	Will not upload file TSTFI23372307 to the portal since upload_file is False


DEBUG:iu_debug:Will not upload file TSTFI23372307 to the portal since upload_file is False


In [205]:
# Take a look at the posting result for RNA output
rna_post_log

{'rna_aggregated_counts_h5ad': ['TSTFI71339318', 201],
 'rna_mtx_tar': ['TSTFI52849784', 201],
 'rna_mtxs_h5ad': ['TSTFI58629611', 201],
 'rna_kb_output': ['TSTFI23372307', 201]}

In [219]:
# Try posting all ATAC output
atac_post_log = t2p_tools.post_all_atac_data_to_portal(terra_data_record=sub_terra_push_series, award=tester_award, lab=tester_lab, igvf_api=igvf_api_sandbox, igvf_utils_api=iu_conn, upload_file=False)

2024-12-03 16:07:45,460:iu_debug:	
IN post().


DEBUG:iu_debug:
IN post().


2024-12-03 16:07:46,392:iu_debug:	Validating the payload against the schema


DEBUG:iu_debug:Validating the payload against the schema


2024-12-03 16:07:46,444:iu_debug:	<<<<<< POST alignment_file record ansuman-satpathy:TSTDS33660419_atac_bam_uniform-pipeline To IGVF database with URL https://api.sandbox.igvf.org/alignment_file and this payload:

{
  "aliases": [
    "ansuman-satpathy:TSTDS33660419_atac_bam_uniform-pipeline"
  ],
  "assembly": "GRCh38",
  "award": "/awards/HG012076/",
  "content_type": "alignments",
  "controlled_access": false,
  "derived_from": [
    "TSTFI68061834",
    "TSTFI78417402",
    "TSTFI18171101",
    "TSTFI58997785"
  ],
  "file_format": "bam",
  "file_set": "TSTDS33660419",
  "file_size": 46817153105,
  "filtered": false,
  "lab": "/labs/ansuman-satpathy/",
  "md5sum": "25dc8ff1cfeee8e742208321b0b88f9d",
  "redacted": false,
  "reference_files": [
    "TSTFI36924773"
  ],
  "submitted_file_name": "gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7ad1a5/submissions/f72bac2f-e428-467c-ac78-4a4919e04103/multiome_pipeline/8aa086d8-3230-4e7d-aa6b-a2e70bfa2aad/call-atac/wf_atac/1ce3aa9a-19db-4ba5

...

DEBUG:iu_debug:Success.


2024-12-03 16:07:53,967:iu_debug:	Object posted with identifier: TSTFI74851839


DEBUG:iu_debug:Object posted with identifier: TSTFI74851839
INFO:iu_post:ansuman-satpathy:TSTDS33660419_atac_filter_fragments_index_uniform-pipeline		TSTFI74851839


2024-12-03 16:07:53,975:iu_debug:	Will not upload file TSTFI74851839 to the portal since upload_file is False


DEBUG:iu_debug:Will not upload file TSTFI74851839 to the portal since upload_file is False


In [220]:
# Take a look at ATAC result posting
atac_post_log

{'atac_bam': ['TSTFI88557142', 409],
 'atac_filter_fragments': ['TSTFI26374507', 201],
 'atac_filter_fragments_index': ['TSTFI74851839', 201]}

In [9]:
# Test upload a file after POSTING
iu_conn.after_submit_file_cloud_upload(profile_id='matrix_file', rec_id='TSTFI71339318')

2024-12-10 15:44:22,858:iu_debug:	
IN upload_file()

2024-12-10 15:44:22,860:iu_debug:	Attempting to generate new file upload credentials
2024-12-10 15:44:23,935:iu_debug:	Success: upload credentials for 'TSTFI71339318' regenerated.
2024-12-10 15:44:23,938:iu_debug:	>>>>>>GET TSTFI71339318 From DACC with URL https://api.sandbox.igvf.org/TSTFI71339318/?format=json&datastore=database
2024-12-10 15:44:25,958:iu_debug:	Uploading file gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7ad1a5/submissions/f72bac2f-e428-467c-ac78-4a4919e04103/multiome_pipeline/8aa086d8-3230-4e7d-aa6b-a2e70bfa2aad/call-rna/wf_rna/91fee12c-3166-4d1c-a441-f51e2e2ed977/call-kb/Dataset510xmultiomLane1.rna.align.kb.hg38.cells_x_genes.total.h5ad to s3://igvf-files-staging/2024/12/03/e9929399-d786-40f6-a48a-889c360395c6/TSTFI71339318.h5ad
2024-12-10 15:56:14,204:iu_debug:	Finished uploading file gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7ad1a5/submissions/f72bac2f-e428-467c-ac78-4a4919e04103/multiome_pipeline/8aa086d8-32

#### Playcodes for POSTing

In [84]:
# Mtx file G cloud location
terra_push_example_mtx = terra_push_example['rna_mtxs_h5ad'].values[2]

# Mtx file md5
terra_push_example_mtx_md5 = api_tools.calculate_gsutil_hash(file_path=terra_push_example_mtx)

In [95]:
# Make payload
mtx_file_payload = {
                        "aliases": ["igvf:terra_rna_h5ad_team2_file_by_iuregister_2"],
                        "award": "/awards/HG012012",
                        "lab": "/labs/j-michael-cherry/",
                        "submitter_comment": "Test submission of a mtx file from Terra",
                        "content_type": "sparse gene count matrix",
                        "file_format": "h5ad",
                        "file_set": "TSTDS54492851/",
                        "md5sum": terra_push_example_mtx_md5,
                        "submitted_file_name": terra_push_example_mtx,
                        "principal_dimension": "table",
                        "secondary_dimensions": ["gene"],
                        "reference_files": ["/reference-files/TSTFI36924773/"]
                    }
mtx_file_payload[Connection.PROFILE_KEY] = "matrix_file"
mtx_file_payload

{'aliases': ['igvf:terra_rna_h5ad_team2_file_by_iuregister_2'],
 'award': '/awards/HG012012',
 'lab': '/labs/j-michael-cherry/',
 'submitter_comment': 'Test submission of a mtx file from Terra',
 'content_type': 'sparse gene count matrix',
 'file_format': 'h5ad',
 'file_set': '/analysis-sets/TSTDS54492851/',
 'md5sum': 'ad6d9e540e6182c5f69d6ee17bb1934e',
 'submitted_file_name': 'gs://fc-secure-de19fd29-2253-41cd-9751-1788cf7ad1a5/submissions/f72bac2f-e428-467c-ac78-4a4919e04103/multiome_pipeline/bae29912-fb64-4614-b198-5f99d27248ef/call-rna/wf_rna/96adc130-7173-4b3f-8162-113618a2c60d/call-kb/Dataset510xmultiomLane3.rna.align.kb.hg38.count_matrix.h5ad',
 'principal_dimension': 'table',
 'secondary_dimensions': ['gene'],
 'reference_files': ['/reference-files/TSTFI36924773/'],
 '_profile': 'matrix_file'}

In [99]:
_schema_property = iu_conn.get_profile_from_payload(mtx_file_payload).properties
iu_post_log = iu_conn.post(mtx_file_payload, upload_file=False, return_original_status_code=True, dry_run=True)

TypeError: Connection.post() got an unexpected keyword argument 'dry_run'

In [100]:
help(iu_conn.post)

Help on method post in module igvf_utils.connection:

post(payload, require_aliases=True, upload_file=True, return_original_status_code=False, truncate_long_strings_in_payload_log=False) method of igvf_utils.connection.Connection instance
    POST a record to the Portal.
    
    Requires that you include in the payload the non-schematic key ``self.PROFILE_KEY`` to
    designate the name of the IGVF object profile that you are submitting to, or the
    actual `@id` property itself.
    
    If the `lab` property isn't present in the payload, then the default will be set to the value
    of the `IGVF_LAB` environment variable. Similarly, if the `award` property isn't present, then the
    default will be set to the value of the `IGVF_AWARD` environment variable.
    
    Before the POST is attempted, any pre-POST hooks are fist called; see the method
    ``self.before_submit_hooks``).  After a successfuly POST, any after-POST submit hooks are
    also run; see the method ``self.after_su